*   To handle files from Google Drive, Colab should have the necessary permissions to mount the drive.

In [ ]:
# import pandas as pd
# # Give permissions to colab handling files from gdrive
# from google.colab import drive
# drive.mount('/content/gdrive')
# # Check if the file exists
# import os
# file_path = '/content/gdrive/My Drive/SentimentBERT/dataset-test.csv'
# if os.path.exists(file_path):
#     print("File exists.")
# else:
#     print("File does not exist.")

* Use the training dataset uploaded to Google Drive to **train** the **RoBERTa** model using **AdamW Optimizer**

In [1]:
import pandas as pd
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW, RobertaConfig # Hugging Face transformers library
from torch.utils.data import Dataset, DataLoader # PyTorch for handling datasets and loading batches
from tqdm import tqdm # Progress bars
import torch # PyTorch functionalities
from sklearn.metrics import accuracy_score # Scikit-learn for evaluating classification accuracy
from torch.optim import AdamW as AdamW_Torch # AdamW Optimizer
from google.colab import drive # Use Google classes

drive.mount('/content/gdrive') # Mount Google Drive using Google Colab to access files stored in Google Drive
df = pd.read_csv("/content/gdrive/My Drive/EcommerceReviews/Datasets/train_dataset.csv") # Load the test dataset from GDrive

# Tokenization
tokenizer = RobertaTokenizer.from_pretrained('roberta-base') # Instantiate a roBERTa tokenizer from the Hugging Face transformers library
# Tokenize the text data using the roBERTa tokenizer
tokenized = tokenizer(list(df['review']), # Convert the column of the DataFrame into a Python list
                      padding=True, # Equal length sequences
                      truncation=True, # Truncate long reviews to fit the maximum allowed sequence length
                      return_tensors='pt') # Tokenizer output in PyTorch format  (PyTorch tensor - multi-dimensional array)

# Number of classes for sentiment classification are the number of unique labels in the 'review_rating' column
num_labels = len(df['review_rating'].unique())

class CustomDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs['input_ids'])

    def __getitem__(self, idx):
        return {'input_ids': self.inputs['input_ids'][idx],
                'attention_mask': self.inputs['attention_mask'][idx],
                'labels': torch.tensor(self.labels[idx] - 1)} # Adjust labels to start from 0

def evaluate_model(model, dataloader, device): # Evaluate model's accuracy on a given dataloader
    model.eval()
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for batch in dataloader:
            inputs = {key: value.to(device) for key, value in batch.items()}
            labels = inputs["labels"]
            outputs = model(**inputs)
            logits = outputs.logits

            _, predicted = torch.max(logits, 1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    return accuracy

# Training dataset
train_dataset = CustomDataset(tokenized, # the output of roBERTa tokenizer
                              list(df['review_rating'])) # Instantiate CustomDataset class
train_dataloader = DataLoader(train_dataset, # The DataLoader will iterate over CustomDataset
                              batch_size=8, # Number of samples in each mini-batch for memory-efficiency
                              shuffle=True # Randomly shuffles the data at the beginning of each epoch, preventing model from learning the order of the data (useful for generalization)
                              ) # Create a PyTorch DataLoader for the entire dataset

# Validation dataset
validation_df = pd.read_csv("/content/gdrive/My Drive/EcommerceReviews/Datasets/validation_dataset.csv") # Load the validation dataset
tokenized_validation = tokenizer(list(validation_df['review']),
                                  padding=True,
                                  truncation=True,
                                  return_tensors='pt') # Tokenize the text data

validation_dataset = CustomDataset(tokenized_validation, list(validation_df['review_rating']))
validation_dataloader = DataLoader(validation_dataset, batch_size=8, shuffle=False)

# Load pre-trained roBERTa model with the correct num_labels
config = RobertaConfig.from_pretrained('roberta-base', num_labels=num_labels)
model = RobertaForSequenceClassification.from_pretrained('roberta-base', config=config)

# Fine-tuning
optimizer = AdamW_Torch(model.parameters(), lr=2e-5) # AdamW optimizer from PyTorch with a learning rate of 2e-5

num_epochs = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Use GPU if it is available
model.to(device)

# Loop over Epochs and Perform training
for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{num_epochs}'): # Batch Loop with Progress Bar
        inputs = {key: value.to(device) for key, value in batch.items()} # Move input data to the selected device
        outputs = model(**inputs)
        loss = outputs.loss # Loss value from the model's output
        train_losses.append(loss.item()) # Append loss values to a list

        optimizer.zero_grad() # Clear the gradients of all optimized parameters
        loss.backward() # Compute gradients of the loss during backpropagation
        optimizer.step() # Update model's parameters based on the computed gradients using the chosen AdamW optimization algorithm

    # Validation loop
    model.eval()
    validation_losses = []

    with torch.no_grad():
        for batch in tqdm(validation_dataloader, desc=f'Validation - Epoch {epoch + 1}/{num_epochs}'):
            inputs = {key: value.to(device) for key, value in batch.items()}
            outputs = model(**inputs)
            validation_losses.append(outputs.loss.item())

    # Assess the accuracy of the trained model on the validation set
    accuracy = evaluate_model(model, validation_dataloader, device)

    print(f'Epoch {epoch + 1}/{num_epochs} - Training Loss: {sum(train_losses) / len(train_losses):.4f} - Validation Loss: {sum(validation_losses) / len(validation_losses):.4f} - Validation Accuracy: {accuracy:.4f}')

# Save the fine-tuned model and tokenizer to a directory
model.save_pretrained('/content/gdrive/My Drive/EcommerceReviews/roBERTa/adamw')
tokenizer.save_pretrained('/content/gdrive/My Drive/EcommerceReviews/roBERTa/adamw')

Mounted at /content/gdrive


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...

Validation - Epoch 2/3: 100%|██████████| 95/95 [00:02<00:00, 36.18it/s]


Epoch 2/3 - Training Loss: 0.9232 - Validation Loss: 0.9335 - Validation Accuracy: 0.6446


Validation - Epoch 3/3: 100%|██████████| 95/95 [00:02<00:00, 35.71it/s]


Epoch 3/3 - Training Loss: 0.7634 - Validation Loss: 1.0822 - Validation Accuracy: 0.5875


('/content/gdrive/My Drive/EcommerceReviews/roBERTa/adamw/tokenizer_config.json',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adamw/special_tokens_map.json',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adamw/vocab.json',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adamw/merges.txt',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adamw/added_tokens.json')

* Use the trained **roBERTa** model (**AdamW Optimizer**) to **predict** ratings for the **test** dataset.

In [3]:
import pandas as pd
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/gdrive')

# Load the test dataset from GDrive
test_df = pd.read_csv("/content/gdrive/My Drive/EcommerceReviews/Datasets/test_dataset.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Use GPU if available

def load_fine_tuned_roberta_model(model_path):
    # Load the fine-tuned model and tokenizer
    model = RobertaForSequenceClassification.from_pretrained(model_path) # Load a pre-trained roBERTa model for sequence classification
    tokenizer = RobertaTokenizer.from_pretrained(model_path) # Load the tokenizer

    model.to(device) # Move model to GPU

    return model, tokenizer

def predict_star_rating(model, tokenizer, new_text, max_seq_length=512):
    # Tokenize and encode the new text
    tokens = tokenizer.tokenize(tokenizer.decode(tokenizer.encode(new_text)))

    # Truncate or split the sequence if it exceeds the maximum allowed length
    if len(tokens) > max_seq_length - 2:
        tokens = tokens[:max_seq_length - 2]

    input_ids = tokenizer.encode(tokens, return_tensors="pt").to(device)  # Tokenize and encode the input text using the roBERTa tokenizer

    # Make predictions
    with torch.no_grad():
        model.eval()  # Set the BERT model to evaluation mode
        logits = model(input_ids)[0]  # Pass the encoded input through the roBERTa model to obtain logits (raw predictions)
        predictions = torch.argmax(logits, dim=1).item()  # Get the predicted class by finding the index with the maximum value in the logits tensor

    # Return the predicted star rating
    return predictions + 1  # Adding 1 because star ratings are 1-indexed not 0-indexed

def process_review_and_predict_rating(review_content, model, tokenizer):
    predicted_rating = predict_star_rating(model, tokenizer, review_content)
    return predicted_rating

# Load the fine-tuned RoBERTa model and tokenizer
model_path = '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adamw'
model, tokenizer = load_fine_tuned_roberta_model(model_path)

# Create a progress bar for the loop
tqdm.pandas()

# Iterate through each row in the DataFrame
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    review_id = row['review_id']
    review_content = row['review']

    # Process the review and predict the star rating
    predicted_rating = process_review_and_predict_rating(review_content, model, tokenizer)

    # Update the 'review_rating' column
    test_df.at[index, 'review_rating'] = predicted_rating

# Save the updated DataFrame back to the CSV file
test_df.to_csv("/content/gdrive/My Drive/EcommerceReviews/Predictions/roberta-adamw-predictions.csv", index=False)


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


100%|██████████| 755/755 [00:11<00:00, 63.36it/s]


* Use the training dataset uploaded to Google Drive to **train** the **roBERTa** model using **Adam Optimizer**

In [4]:
import pandas as pd
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW, RobertaConfig # Hugging Face transformers library
from torch.utils.data import Dataset, DataLoader # PyTorch for handling datasets and loading batches
from tqdm import tqdm # Progress bars
import torch # PyTorch functionalities
from sklearn.metrics import accuracy_score # Scikit-learn for evaluating classification accuracy
from torch.optim import Adam as Adam_Torch # Adam Optimizer
from google.colab import drive # Use Google classes

drive.mount('/content/gdrive') # Mount Google Drive using Google Colab to access files stored in Google Drive
df = pd.read_csv("/content/gdrive/My Drive/EcommerceReviews/Datasets/train_dataset.csv") # Load the test dataset from GDrive

# Tokenization
tokenizer = RobertaTokenizer.from_pretrained('roberta-base') # Instantiate a roBERTa tokenizer from the Hugging Face transformers library
# Tokenize the text data using the roBERTa tokenizer
tokenized = tokenizer(list(df['review']), # Convert the column of the DataFrame into a Python list
                      padding=True, # Equal length sequences
                      truncation=True, # Truncate long reviews to fit the maximum allowed sequence length
                      return_tensors='pt') # Tokenizer output in PyTorch format  (PyTorch tensor - multi-dimensional array)

# Number of classes for sentiment classification are the number of unique labels in the 'review_rating' column
num_labels = len(df['review_rating'].unique())

class CustomDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs['input_ids'])

    def __getitem__(self, idx):
        return {'input_ids': self.inputs['input_ids'][idx],
                'attention_mask': self.inputs['attention_mask'][idx],
                'labels': torch.tensor(self.labels[idx] - 1)} # Adjust labels to start from 0

def evaluate_model(model, dataloader, device): # Evaluate model's accuracy on a given dataloader
    model.eval()
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for batch in dataloader:
            inputs = {key: value.to(device) for key, value in batch.items()}
            labels = inputs["labels"]
            outputs = model(**inputs)
            logits = outputs.logits

            _, predicted = torch.max(logits, 1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    return accuracy

# Training dataset
train_dataset = CustomDataset(tokenized, # the output of roBERTa tokenizer
                              list(df['review_rating'])) # Instantiate CustomDataset class
train_dataloader = DataLoader(train_dataset, # The DataLoader will iterate over CustomDataset
                              batch_size=8, # Number of samples in each mini-batch for memory-efficiency
                              shuffle=True # Randomly shuffles the data at the beginning of each epoch, preventing model from learning the order of the data (useful for generalization)
                              ) # Create a PyTorch DataLoader for the entire dataset

# Validation dataset
validation_df = pd.read_csv("/content/gdrive/My Drive/EcommerceReviews/Datasets/validation_dataset.csv") # Load the validation dataset
tokenized_validation = tokenizer(list(validation_df['review']),
                                  padding=True,
                                  truncation=True,
                                  return_tensors='pt') # Tokenize the text data

validation_dataset = CustomDataset(tokenized_validation, list(validation_df['review_rating']))
validation_dataloader = DataLoader(validation_dataset, batch_size=8, shuffle=False)

# Load pre-trained roBERTa model with the correct num_labels
config = RobertaConfig.from_pretrained('roberta-base', num_labels=num_labels)
model = RobertaForSequenceClassification.from_pretrained('roberta-base', config=config)

# Fine-tuning
optimizer = Adam_Torch(model.parameters(), lr=2e-5) # Adam optimizer from PyTorch with a learning rate of 2e-5

num_epochs = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Use GPU if it is available
model.to(device)

# Loop over Epochs and Perform training
for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{num_epochs}'): # Batch Loop with Progress Bar
        inputs = {key: value.to(device) for key, value in batch.items()} # Move input data to the selected device
        outputs = model(**inputs)
        loss = outputs.loss # Loss value from the model's output
        train_losses.append(loss.item()) # Append loss values to a list

        optimizer.zero_grad() # Clear the gradients of all optimized parameters
        loss.backward() # Compute gradients of the loss during backpropagation
        optimizer.step() # Update model's parameters based on the computed gradients using the chosen Adam optimization algorithm

    # Validation loop
    model.eval()
    validation_losses = []

    with torch.no_grad():
        for batch in tqdm(validation_dataloader, desc=f'Validation - Epoch {epoch + 1}/{num_epochs}'):
            inputs = {key: value.to(device) for key, value in batch.items()}
            outputs = model(**inputs)
            validation_losses.append(outputs.loss.item())

    # Assess the accuracy of the trained model on the validation set
    accuracy = evaluate_model(model, validation_dataloader, device)

    print(f'Epoch {epoch + 1}/{num_epochs} - Training Loss: {sum(train_losses) / len(train_losses):.4f} - Validation Loss: {sum(validation_losses) / len(validation_losses):.4f} - Validation Accuracy: {accuracy:.4f}')

# Save the fine-tuned model and tokenizer to a directory
model.save_pretrained('/content/gdrive/My Drive/EcommerceReviews/roBERTa/adam')
tokenizer.save_pretrained('/content/gdrive/My Drive/EcommerceReviews/roBERTa/adam')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.out_proj.weight', 'classifier.out_proj.bias', 'classifier.dense.weight', 'classifier.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Validation - Epoch 1/3: 100%|██████████| 95/95 [00:02<00:00, 34.97it/s]


Epoch 1/3 - Training Loss: 1.2080 - Validation Loss: 1.0537 - Validation Accuracy: 0.5703


Validation - Epoch 2/3: 100%|██████████| 95/95 [00:02<00:00, 35.81it/s]


Epoch 2/3 - Training Loss: 0.9372 - Validation Loss: 0.9591 - Validation Accuracy: 0.5703


Validation - Epoch 3/3: 100%|██████████| 95/95 [00:02<00:00, 36.16it/s]


Epoch 3/3 - Training Loss: 0.7644 - Validation Loss: 0.9966 - Validation Accuracy: 0.6260


('/content/gdrive/My Drive/EcommerceReviews/roBERTa/adam/tokenizer_config.json',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adam/special_tokens_map.json',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adam/vocab.json',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adam/merges.txt',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adam/added_tokens.json')

* Use the trained **roBERTa** model (**Adam Optimizer**) to **predict** ratings for the **test** dataset.

In [5]:
import pandas as pd
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/gdrive')

# Load the test dataset from GDrive
test_df = pd.read_csv("/content/gdrive/My Drive/EcommerceReviews/Datasets/test_dataset.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Use GPU if available

def load_fine_tuned_roberta_model(model_path):
    # Load the fine-tuned model and tokenizer
    model = RobertaForSequenceClassification.from_pretrained(model_path) # Load a pre-trained roBERTa model for sequence classification
    tokenizer = RobertaTokenizer.from_pretrained(model_path) # Load the tokenizer

    model.to(device) # Move model to GPU

    return model, tokenizer

def predict_star_rating(model, tokenizer, new_text, max_seq_length=512):
    # Tokenize and encode the new text
    tokens = tokenizer.tokenize(tokenizer.decode(tokenizer.encode(new_text)))

    # Truncate or split the sequence if it exceeds the maximum allowed length
    if len(tokens) > max_seq_length - 2:
        tokens = tokens[:max_seq_length - 2]

    input_ids = tokenizer.encode(tokens, return_tensors="pt").to(device)  # Tokenize and encode the input text using the roBERTa tokenizer

    # Make predictions
    with torch.no_grad():
        model.eval()  # Set the BERT model to evaluation mode
        logits = model(input_ids)[0]  # Pass the encoded input through the roBERTa model to obtain logits (raw predictions)
        predictions = torch.argmax(logits, dim=1).item()  # Get the predicted class by finding the index with the maximum value in the logits tensor

    # Return the predicted star rating
    return predictions + 1  # Adding 1 because star ratings are 1-indexed not 0-indexed

def process_review_and_predict_rating(review_content, model, tokenizer):
    predicted_rating = predict_star_rating(model, tokenizer, review_content)
    return predicted_rating

# Load the fine-tuned RoBERTa model and tokenizer
model_path = '/content/gdrive/My Drive/EcommerceReviews/roBERTa/adam'
model, tokenizer = load_fine_tuned_roberta_model(model_path)

# Create a progress bar for the loop
tqdm.pandas()

# Iterate through each row in the DataFrame
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    review_id = row['review_id']
    review_content = row['review']

    # Process the review and predict the star rating
    predicted_rating = process_review_and_predict_rating(review_content, model, tokenizer)

    # Update the 'review_rating' column
    test_df.at[index, 'review_rating'] = predicted_rating

# Save the updated DataFrame back to the CSV file
test_df.to_csv("/content/gdrive/My Drive/EcommerceReviews/Predictions/roberta-adam-predictions.csv", index=False)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


100%|██████████| 755/755 [00:11<00:00, 65.62it/s]


* Use the training dataset uploaded to Google Drive to **train** the **roBERTa** model using **Stochastic Gradient Descent (SGD) Optimizer**

In [6]:
import pandas as pd
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW, RobertaConfig
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torch
from sklearn.metrics import accuracy_score
from torch.optim import SGD
from google.colab import drive

drive.mount('/content/gdrive')

df = pd.read_csv("/content/gdrive/My Drive/EcommerceReviews/Datasets/train_dataset.csv")

# Tokenization
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')  # Use RoBERTa tokenizer
tokenized = tokenizer(list(df['review']), padding=True, truncation=True, return_tensors='pt')

# Number of classes for sentiment classification are the number of unique labels in the 'review_rating' column
num_labels = len(df['review_rating'].unique())

class CustomDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs['input_ids'])

    def __getitem__(self, idx):
        return {'input_ids': self.inputs['input_ids'][idx],
                'attention_mask': self.inputs['attention_mask'][idx],
                'labels': torch.tensor(self.labels[idx] - 1)}  # Adjust labels to start from 0

def evaluate_model(model, dataloader, device):
    model.eval()
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for batch in dataloader:
            inputs = {key: value.to(device) for key, value in batch.items()}
            labels = inputs["labels"]
            outputs = model(**inputs)
            logits = outputs.logits

            _, predicted = torch.max(logits, 1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    return accuracy

# Training dataset
train_dataset = CustomDataset(tokenized, list(df['review_rating']))
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Validation dataset
validation_df = pd.read_csv("/content/gdrive/My Drive/EcommerceReviews/Datasets/validation_dataset.csv")
tokenized_validation = tokenizer(list(validation_df['review']), padding=True, truncation=True, return_tensors='pt')
validation_dataset = CustomDataset(tokenized_validation, list(validation_df['review_rating']))
validation_dataloader = DataLoader(validation_dataset, batch_size=8, shuffle=False)

# Load pre-trained RoBERTa model with the correct num_labels
config = RobertaConfig.from_pretrained('roberta-base', num_labels=num_labels)
model = RobertaForSequenceClassification.from_pretrained('roberta-base', config=config)

# Fine-tuning
optimizer = SGD(model.parameters(), lr=2e-5, momentum=0.9)

num_epochs = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Loop over Epochs and Perform training
for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{num_epochs}'):
        inputs = {key: value.to(device) for key, value in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        train_losses.append(loss.item())

        model.zero_grad()
        loss.backward()

        for param in model.parameters():
            param.data -= 2e-5 * param.grad

    model.eval()
    validation_losses = []

    with torch.no_grad():
        for batch in tqdm(validation_dataloader, desc=f'Validation - Epoch {epoch + 1}/{num_epochs}'):
            inputs = {key: value.to(device) for key, value in batch.items()}
            outputs = model(**inputs)
            validation_losses.append(outputs.loss.item())

    accuracy = evaluate_model(model, validation_dataloader, device)

    print(f'Epoch {epoch + 1}/{num_epochs} - Training Loss: {sum(train_losses) / len(train_losses):.4f} - Validation Loss: {sum(validation_losses) / len(validation_losses):.4f} - Validation Accuracy: {accuracy:.4f}')

# Save the fine-tuned model and tokenizer to a directory
model.save_pretrained('/content/gdrive/My Drive/EcommerceReviews/roBERTa/sgd')
tokenizer.save_pretrained('/content/gdrive/My Drive/EcommerceReviews/roBERTa/sgd')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.out_proj.weight', 'classifier.out_proj.bias', 'classifier.dense.weight', 'classifier.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Validation - Epoch 1/3: 100%|██████████| 95/95 [00:02<00:00, 35.63it/s]


Epoch 1/3 - Training Loss: 1.6175 - Validation Loss: 1.5975 - Validation Accuracy: 0.2666


Validation - Epoch 2/3: 100%|██████████| 95/95 [00:02<00:00, 35.01it/s]


Epoch 2/3 - Training Loss: 1.6103 - Validation Loss: 1.5942 - Validation Accuracy: 0.2772


Validation - Epoch 3/3: 100%|██████████| 95/95 [00:02<00:00, 35.86it/s]


Epoch 3/3 - Training Loss: 1.6051 - Validation Loss: 1.5922 - Validation Accuracy: 0.2772


('/content/gdrive/My Drive/EcommerceReviews/roBERTa/sgd/tokenizer_config.json',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/sgd/special_tokens_map.json',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/sgd/vocab.json',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/sgd/merges.txt',
 '/content/gdrive/My Drive/EcommerceReviews/roBERTa/sgd/added_tokens.json')

* Use the trained **roBERTa** model (**SGD Optimizer**) to **predict** ratings for the **test** dataset.

In [7]:
import pandas as pd
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/gdrive')

# Load the test dataset from GDrive
test_df = pd.read_csv("/content/gdrive/My Drive/EcommerceReviews/Datasets/test_dataset.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Use GPU if available

def load_fine_tuned_roberta_model(model_path):
    # Load the fine-tuned model and tokenizer
    model = RobertaForSequenceClassification.from_pretrained(model_path) # Load a pre-trained roBERTa model for sequence classification
    tokenizer = RobertaTokenizer.from_pretrained(model_path) # Load the tokenizer

    model.to(device) # Move model to GPU

    return model, tokenizer

def predict_star_rating(model, tokenizer, new_text, max_seq_length=512):
    # Tokenize and encode the new text
    tokens = tokenizer.tokenize(tokenizer.decode(tokenizer.encode(new_text)))

    # Truncate or split the sequence if it exceeds the maximum allowed length
    if len(tokens) > max_seq_length - 2:
        tokens = tokens[:max_seq_length - 2]

    input_ids = tokenizer.encode(tokens, return_tensors="pt").to(device)  # Tokenize and encode the input text using the roBERTa tokenizer

    # Make predictions
    with torch.no_grad():
        model.eval()  # Set the BERT model to evaluation mode
        logits = model(input_ids)[0]  # Pass the encoded input through the roBERTa model to obtain logits (raw predictions)
        predictions = torch.argmax(logits, dim=1).item()  # Get the predicted class by finding the index with the maximum value in the logits tensor

    # Return the predicted star rating
    return predictions + 1  # Adding 1 because star ratings are 1-indexed not 0-indexed

def process_review_and_predict_rating(review_content, model, tokenizer):
    predicted_rating = predict_star_rating(model, tokenizer, review_content)
    return predicted_rating

# Load the fine-tuned RoBERTa model and tokenizer
model_path = '/content/gdrive/My Drive/EcommerceReviews/roBERTa/sgd'
model, tokenizer = load_fine_tuned_roberta_model(model_path)

# Create a progress bar for the loop
tqdm.pandas()

# Iterate through each row in the DataFrame
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    review_id = row['review_id']
    review_content = row['review']

    # Process the review and predict the star rating
    predicted_rating = process_review_and_predict_rating(review_content, model, tokenizer)

    # Update the 'review_rating' column
    test_df.at[index, 'review_rating'] = predicted_rating

# Save the updated DataFrame back to the CSV file
test_df.to_csv("/content/gdrive/My Drive/EcommerceReviews/Predictions/roberta-sgd-predictions.csv", index=False)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


100%|██████████| 755/755 [00:11<00:00, 66.29it/s]
